# SBD POC3 — dummy data run

Same pipeline as `sbd_poc3_local_fixture.ipynb`, but fed synthetic rows from
this repo's own `generate_standard_simulation_rows` instead of a real fixture
file. Lets us exercise the loader -> `Portfolio.simulate()` path before
Sebastian's data or the UC grant lands — no fixture file needed.

In [1]:
from dataclasses import replace

from replenishment.io_ import (
    generate_standard_simulation_rows,
    standard_simulation_rows_to_dataframe,
)

# Per-item spec: variable lead times, unit economics, and supplier MoQs.
# Costs derive from unit economics:
#   holding_cost_per_unit  = HOLDING_RATE * unit_cost   (carrying cost per period)
#   stockout_cost_per_unit = price - unit_cost          (lost margin per missed unit)
# moq = supplier minimum order quantity: any positive order is floored to it
# (a "don't order" period stays 0 — MoQ never creates an order).
# Mean demand is ~18/period, so MoQ ranges from "barely binding" (F: 10)
# to "forces ~8 periods of stock per order" (G: 150).
ITEM_SPECS = {
    "A": dict(lead_time=2, unit_cost=4.0, price=6.0, moq=50),
    "B": dict(lead_time=3, unit_cost=8.0, price=10.0, moq=20),
    "C": dict(lead_time=4, unit_cost=2.5, price=5.0, moq=100),
    "D": dict(lead_time=5, unit_cost=12.0, price=18.0, moq=25),
    "E": dict(lead_time=6, unit_cost=6.0, price=7.5, moq=60),
    "F": dict(lead_time=7, unit_cost=20.0, price=32.0, moq=10),   # MoQ below mean demand: rarely binds
    "G": dict(lead_time=8, unit_cost=3.0, price=3.6, moq=150),    # thin margin, long lead time, huge MoQ
    "H": dict(lead_time=10, unit_cost=15.0, price=25.0, moq=40),
}
HOLDING_RATE = 0.02

rows = []
for i, (uid, spec) in enumerate(ITEM_SPECS.items()):
    item_rows = generate_standard_simulation_rows(
        n_unique_ids=1,
        periods=120,
        history_mean=18,
        history_std=4,
        forecast_mean=18,
        forecast_std=3,
        lead_time=spec["lead_time"],
        holding_cost_per_unit=round(HOLDING_RATE * spec["unit_cost"], 4),
        stockout_cost_per_unit=round(spec["price"] - spec["unit_cost"], 4),
        order_cost_per_order=12.5,
        seed=7 + i,
    )
    # generator labels every single-item call "A" — relabel to this item's id
    rows.extend(replace(r, unique_id=uid) for r in item_rows)

df = standard_simulation_rows_to_dataframe(rows, library="pandas")
df["current_stock"] = 20
df = df.drop(columns=["is_forecast"])
df.groupby("unique_id")[
    ["lead_time", "holding_cost_per_unit", "stockout_cost_per_unit", "order_cost_per_order"]
].first().assign(moq=lambda d: d.index.map(lambda uid: ITEM_SPECS[uid]["moq"]))

,lead_time,holding_cost_per_unit,stockout_cost_per_unit,order_cost_per_order,moq
unique_id,,,,,
A,2,0.08,2.0,12.5,50
B,3,0.16,2.0,12.5,20
C,4,0.05,2.5,12.5,100
D,5,0.24,6.0,12.5,25
E,6,0.12,1.5,12.5,60
F,7,0.40,12.0,12.5,10
G,8,0.06,0.6,12.5,150
H,10,0.30,10.0,12.5,40


In [2]:
# Round-trip through the loader — this is the path real SBD data will take.
from replenishment import Portfolio

port = Portfolio.from_dataframe(df)
len(port.rows)

960

## Data contract — forecast-driven optimization (lead-time forecast)

To run `Portfolio.simulate(factor=..., horizon=lead_times)`, the input
dataframe must satisfy the same standard-simulation shape the loader
(`Portfolio.from_dataframe` / `standard_simulation_rows_from_dataframe`)
already enforces:

| column | type | notes |
|---|---|---|
| `unique_id` | str | article key |
| `ds` | date str | one row per period, sorted per article |
| `forecast` | int | per-period point forecast (drives ordering) |
| `actuals` (or `history` / `demand`) | int | realized demand; feeds forecast-error safety stock and the simulated demand stream |
| `lead_time` | int | **constant per unique_id** |
| `holding_cost_per_unit` | float | **constant per unique_id** |
| `stockout_cost_per_unit` | float | **constant per unique_id** |
| `order_cost_per_order` | float | **constant per unique_id** |
| `current_stock` | int | **constant per unique_id**; starting stock when all rows are forecast rows |
| `initial_on_hand` (or `initial_demand`) | int | **constant per unique_id**; starting stock for history-driven backtests |

Simulate-time knobs (arguments, **not** columns) — each a scalar or a
per-article `{unique_id: value}` mapping:

- `factor` — safety-stock z-value
- `method` — e.g. `"sqrt_horizon"` (needs actuals)
- `horizon` — the forecast-driven lever: set to `lead_time` so the
  order-up-to target covers the whole exposure window
- `mode` — `"base_stock"` (order-up-to, default) or `"rop"` (reorder point)
- `moq`, `review_period`, `actuals_override`

**How the two methods differ mechanically.** Both build the same
`ReplenishmentPolicy` with an `OrderUpToTrigger`; the target is
`forecast.sum_over(period + lead_time, forecast_horizon) + safety_stock`.

- *Simulation (point-forecast) method*: `horizon=1` — the target only
  covers **one period** past the lead time. With `lead_time=5`, every order
  protects 1 period of demand while 5 periods of demand drain stock before it
  arrives: chronic under-ordering.
- *Forecast-driven method*: `horizon=lead_time` — the target covers the
  full lead-time window of forecast demand, and `sqrt_horizon` safety stock
  scales as `factor * error * sqrt(lead_time + horizon)`.

In [3]:
# Dummy df already satisfies the contract — assert it explicitly so this cell
# doubles as executable documentation of the requirements.
REQUIRED_COLUMNS = {
    "unique_id", "ds", "forecast", "actuals",
    "lead_time", "holding_cost_per_unit", "stockout_cost_per_unit",
    "order_cost_per_order", "current_stock", "initial_on_hand",
}
CONSTANT_PER_ARTICLE = [
    "lead_time", "holding_cost_per_unit", "stockout_cost_per_unit",
    "order_cost_per_order", "current_stock", "initial_on_hand",
]

missing = REQUIRED_COLUMNS - set(df.columns)
assert not missing, f"contract violation — missing columns: {missing}"
per_article_nunique = df.groupby("unique_id")[CONSTANT_PER_ARTICLE].nunique()
assert (per_article_nunique == 1).all().all(), "contract violation — non-constant per-article column"
print(f"contract OK: {df['unique_id'].nunique()} articles x {df.groupby('unique_id').size().iloc[0]} periods")

contract OK: 8 articles x 120 periods


In [4]:
# Per-item lead times → per-item forecast horizons; per-item MoQs → order floors.
# (Every knob accepts a scalar or a {unique_id: value} mapping.)
LEAD_TIMES = {uid: spec["lead_time"] for uid, spec in ITEM_SPECS.items()}
MOQS = {uid: spec["moq"] for uid, spec in ITEM_SPECS.items()}

# Simulation (point-forecast) method: horizon defaults to 1.
sim_res = port.simulate(factor=1.65, moq=MOQS)

# Forecast-driven method: each item's order-up-to target covers its own lead-time window.
fd_res = port.simulate(factor=1.65, horizon=LEAD_TIMES, moq=MOQS)
len(sim_res), len(fd_res)

(8, 8)

In [5]:
import pandas as pd

comparison = pd.concat(
    {"simulation": sim_res.summary_frame(), "forecast_driven": fd_res.summary_frame()},
    axis=1,
).sort_index(axis=1)
comparison.insert(0, ("item", "lead_time"), pd.Series(LEAD_TIMES))
comparison.insert(1, ("item", "moq"), pd.Series(MOQS))
comparison.insert(
    2,
    ("item", "margin_per_unit"),
    pd.Series({uid: s["price"] - s["unit_cost"] for uid, s in ITEM_SPECS.items()}),
)
comparison[("delta", "fill_rate")] = (
    comparison[("forecast_driven", "fill_rate")] - comparison[("simulation", "fill_rate")]
)
comparison[("delta", "total_cost")] = (
    comparison[("forecast_driven", "total_cost")] - comparison[("simulation", "total_cost")]
)
comparison.loc["MEAN"] = comparison.mean()
comparison.round(2)

item                         forecast_driven            \
          lead_time     moq margin_per_unit     avg_on_hand fill_rate   
unique_id                                                               
A              2.00   50.00            2.00           38.33      1.00   
B              3.00   20.00            2.00           26.72      0.98   
C              4.00  100.00            2.50           63.19      0.98   
D              5.00   25.00            6.00           36.98      0.98   
E              6.00   60.00            1.50           57.52      0.96   
F              7.00   10.00           12.00           28.95      0.95   
G              8.00  150.00            0.60          100.98      0.95   
H             10.00   40.00           10.00           48.34      0.93   
MEAN           5.62   56.88            4.58           50.13      0.97   

                                                               simulation  \
          holding_cost ordering_cost stockout_cost total_cost avg_on_hand   
unique_id                                                                   
A               368.00        550.00         14.00     932.00       21.62   
B               513.12       1312.50         80.00    1905.62        2.70   
C               379.15        287.50        120.00     786.65       31.05   
D              1065.12       1037.50        288.00    2390.62        3.24   
E               828.24        450.00        118.50    1396.74       10.33   
F              1389.60       1437.50       1452.00    4279.10        0.35   
G               727.08        200.00         68.40     995.48       39.93   
H              1740.30        650.00       1580.00    3970.30        4.47   
MEAN            876.33        740.62        465.11    2082.06       14.21   

                                                                         \
          fill_rate holding_cost ordering_cost stockout_cost total_cost   
unique_id                                                                 
A              0.96       207.52        525.00        186.00     918.52   
B              0.72        51.84       1000.00       1218.00    2269.84   
C              0.77       186.30        212.50       1290.00    1688.80   
D              0.54        93.36        575.00       5706.00    6374.36   
E              0.48       148.80        225.00       1687.50    2061.30   
F              0.34        16.80        825.00      17640.00   18481.80   
G              0.60       287.52        125.00        539.40     951.92   
H              0.36       160.80        262.50      14200.00   14623.30   
MEAN           0.60       144.12        468.75       5308.36    5921.23   

              delta             
          fill_rate total_cost  
unique_id                       
A              0.04      13.48  
B              0.26    -364.22  
C              0.21    -902.15  
D              0.43   -3983.74  
E              0.48    -664.56  
F              0.61  -14202.70  
G              0.35      43.56  
H              0.57  -10653.00  
MEAN           0.37   -3839.17

## Compare and contrast

With variable lead times (2→10), per-item unit economics, and per-item MoQs:

| item | lead_time | moq | margin/unit | sim fill | fd fill | Δ total cost (fd − sim) |
|---|---|---|---|---|---|---|
| A | 2 | 50 | 2.00 | 0.96 | 1.00 | +13 |
| G | 8 | 150 | 0.60 | 0.60 | 0.95 | +44 |
| D | 5 | 25 | 6.00 | 0.54 | 0.98 | −3,984 |
| F | 7 | 10 | 12.00 | 0.34 | 0.95 | −14,203 |
| H | 10 | 40 | 10.00 | 0.36 | 0.93 | −10,653 |
| **MEAN** | 5.6 | 56.9 | 4.58 | 0.60 | 0.97 | **−3,839** |

**MoQ acts as accidental safety stock for the point-forecast method.** Every
positive order gets floored to the supplier minimum, so the chronically
under-ordering point-forecast method now overshoots on each order: sim fill
rates jump (A 0.80 → 0.96, G 0.30 → 0.60 vs. the no-MoQ run) without any
change to its one-period target. The bigger the MoQ relative to demand
(~18/period), the bigger the free buffer.

**A binding MoQ compresses the gap between the two methods.** A (moq 50) and
G (moq 150) flip to a small *positive* delta: the forecast-driven method must
also swallow the floor, and since it already orders close to a full
lead-time window, MoQ just piles on holding cost (G carries ~101 units on
hand vs. ~40 for sim). Where MoQ barely binds (F, moq 10 below mean demand),
the no-MoQ story survives intact: fd wins by ~14k because the point-forecast
method still starves a long-lead-time, fat-margin item.

**Ordering cost falls under MoQ.** Floored orders cover more periods, so both
methods order less often (sim ordering cost drops from ~1,463 to ~469 mean).
On real SBD data this is the lever to watch: items whose MoQ is large
relative to weekly demand will look "fine" under the naive method for fill
rate, while quietly carrying MoQ-driven excess stock. The comparison to make
there is holding cost against margin, not fill rate alone.